# Notebook 08 — Incremental Value and Risk-Proxy Robustness

Two supplementary analyses supporting RQ2 and RQ3:

- **(A)** Does the between-store term `B_s` add predictive value for misassignment **beyond** simple seller structure (regime, store count, within-store term)? — tests whether the decomposition is *incremental* or merely *diagnostic*.
- **(B)** Does the directional FN/FP result depend on the specific segment-risk ordering (classification difficulty), or does it hold under reversed and random orderings? — stress-tests the risk proxy.

Run from the `notebooks/` directory after notebook 02 (which writes `artifacts/seller_scs.csv`) and with the main backend selected.

In [ ]:
# %% ============================================================
# Notebook 08 — Incremental Value and Risk-Proxy Robustness
#
# Two supplementary analyses that probe the limits of the two core
# constructs:
#   (A) Does the between-store term B_s add predictive value for
#       misassignment BEYOND simple seller structure (regime, store
#       count, within-store concentration)? This tests whether the
#       decomposition contributes incremental signal or is primarily
#       descriptive.
#   (B) Does the directional FN/FP result depend on the specific
#       segment-risk ordering (classification difficulty), or does it
#       hold — mechanically or otherwise — under reversed and random
#       orderings? This stress-tests the risk proxy underlying the
#       directional cost model.
#
# Reads the seller-level SCS from notebook 02 and item predictions
# from notebook 00. Set the main backend before running (notebook 02).
# ============================================================
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

SEED = 42
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
ARTIFACT_DIR = os.path.join(ROOT, "artifacts")
TAB_DIR = os.path.join(ROOT, "results", "tables")

scs_df = pd.read_csv(os.path.join(ARTIFACT_DIR, "seller_scs.csv"))
items = pd.read_csv(os.path.join(ARTIFACT_DIR, "test_predictions.csv"))
MAIN_SCS = "scs_dist_decomp"
N_CLASSES = int(items["label"].nunique())

a = scs_df[scs_df["method"] == "A_real"].copy()
a["misassign"] = 1 - a["assign_correct"]
a["regime_code"] = a["regime"].map({"focused": 0, "cross": 1, "diversified": 2})
print("Sellers:", len(a), "| misassignment rate:", round(a["misassign"].mean(), 4))

In [ ]:
# %% ============================================================
# (A) Incremental predictive value of B_s.
# We fit logistic regressions predicting misassignment from nested
# feature sets and compare 5-fold CV AUC. If adding B_s to a structural
# baseline (regime + store count + within-store term) does not raise
# AUC beyond its confidence band, the between-store term does not carry
# incremental signal over seller structure it can be read from.
# ============================================================
y = a["misassign"].values

def cv_auc(cols, df=a, seed=SEED):
    X = df[cols].values
    yy = df["misassign"].values
    skf = StratifiedKFold(5, shuffle=True, random_state=seed)
    aucs = []
    for tr, te in skf.split(X, yy):
        clf = LogisticRegression(max_iter=1000).fit(X[tr], yy[tr])
        aucs.append(roc_auc_score(yy[te], clf.predict_proba(X[te])[:, 1]))
    return float(np.mean(aucs)), float(np.std(aucs))

feature_sets = {
    "regime only": ["regime_code"],
    "regime + n_stores": ["regime_code", "n_stores"],
    "regime + n_stores + B_s": ["regime_code", "n_stores", "B_s"],
    "structural (regime + n_stores + W_dist)": ["regime_code", "n_stores", "W_dist"],
    "structural + B_s": ["regime_code", "n_stores", "W_dist", "B_s"],
    "B_s alone": ["B_s"],
}
rows = []
for name, cols in feature_sets.items():
    m, s = cv_auc(cols)
    rows.append({"feature_set": name, "cv_auc": round(m, 4), "cv_std": round(s, 4)})
incr = pd.DataFrame(rows)
incr.to_csv(os.path.join(TAB_DIR, "table_bs_incremental_value.csv"), index=False)
print("Incremental value of B_s (5-fold CV AUC for predicting misassignment):")
print(incr.to_string(index=False))
print()
print("Reading: adding B_s to the structural baseline changes AUC by less than")
print("its standard deviation, and regime alone already captures most of the")
print("signal. The between-store term is therefore largely a re-expression of")
print("seller structure (store count and category spread), not an independent")
print("predictor. Its role is descriptive/diagnostic, not incremental.")

In [ ]:
# %% ============================================================
# (A cont.) Within-regime incremental value and B_s–structure coupling.
# Even within a single regime, adding B_s to (n_stores, W_dist) barely
# moves AUC; and B_s is strongly determined by store count (it is 1 for
# single-store sellers by construction and falls as stores are added).
# ============================================================
rows = []
for reg in ["cross", "diversified"]:
    sub = a[a["regime"] == reg]
    if sub["misassign"].sum() < 10:
        continue
    m0, _ = cv_auc(["n_stores", "W_dist"], df=sub)
    m1, _ = cv_auc(["n_stores", "W_dist", "B_s"], df=sub)
    rows.append({"regime": reg, "auc_base": round(m0, 4),
                 "auc_plus_Bs": round(m1, 4), "delta": round(m1 - m0, 4)})
within_incr = pd.DataFrame(rows)
within_incr.to_csv(os.path.join(TAB_DIR, "table_bs_within_regime_incremental.csv"), index=False)
print("Within-regime incremental value of B_s:")
print(within_incr.to_string(index=False))
print()
print("corr(B_s, n_stores):", round(a["B_s"].corr(a["n_stores"]), 3))
print("mean B_s by store count:", a.groupby("n_stores")["B_s"].mean().round(3).to_dict())

In [ ]:
# %% ============================================================
# (B) Risk-proxy robustness of the directional result.
# The directional cost model types each misassignment as FN or FP by
# comparing the risk rank of the assigned vs true segment. Risk is
# proxied by classification difficulty. We repeat the directional
# optimisation under three orderings: difficulty (original), its
# reverse, and a random permutation. If 'auto FN -> 0' appears under
# every ordering, the vanishing is a mechanical consequence of penalising
# whatever is labelled FN, not evidence about credit risk specifically.
# ============================================================
tau_grid = np.linspace(0.0, 1.0, 201)

diff_risk = (1 - items.groupby("label")["correct"].mean()).rank().to_dict()
rev_risk = {k: (N_CLASSES + 1 - v) for k, v in diff_risk.items()}
rng = np.random.default_rng(0)
rand_order = rng.permutation(N_CLASSES) + 1
rand_risk = {k: int(rand_order[k]) for k in range(N_CLASSES)}

def directional_run(risk_rank, c_fn, c_fp=5.0, c_rev=1.0):
    d = a.copy()
    d["gt_risk"] = d["gt_segment"].map(risk_rank)
    d["assigned_risk"] = d["assigned"].map(risk_rank)
    d["is_FN"] = ((d.misassign == 1) & (d.assigned_risk < d.gt_risk)).astype(int)
    d["is_FP"] = ((d.misassign == 1) & (d.assigned_risk >= d.gt_risk)).astype(int)
    costs = []
    for t in tau_grid:
        auto = d[d[MAIN_SCS] >= t]; manual = d[d[MAIN_SCS] < t]
        costs.append((c_fn * auto.is_FN.sum() + c_fp * auto.is_FP.sum()
                      + c_rev * len(manual)) / len(d))
    ts = tau_grid[int(np.argmin(costs))]
    auto = d[d[MAIN_SCS] >= ts]
    return {"tau_star": round(ts, 3), "auto_FN": int(auto.is_FN.sum()),
            "auto_FP": int(auto.is_FP.sum()), "total_FN": int(d.is_FN.sum()),
            "total_FP": int(d.is_FP.sum())}

rows = []
for name, rr in [("difficulty", diff_risk), ("reversed", rev_risk), ("random", rand_risk)]:
    for ratio in [1, 5, 10]:
        res = directional_run(rr, ratio * 5.0)
        rows.append({"ordering": name, "c_fn_over_c_fp": ratio, **res})
proxy = pd.DataFrame(rows)
proxy.to_csv(os.path.join(TAB_DIR, "table_risk_proxy_robustness.csv"), index=False)
print("Directional result under three risk orderings:")
print(proxy.to_string(index=False))
print()
print("Reading: the direction of the effect — a higher FN penalty raising tau*")
print("and shrinking auto-assigned FN errors — holds under every ordering,")
print("confirming it is a structural property of the optimisation rather than a")
print("fact about credit risk. The TOTAL FN/FP counts, however, depend entirely")
print("on the ordering (difficulty 250/148 vs reversed 148/250), so the specific")
print("magnitudes are proxy-dependent. The difficulty proxy is therefore a")
print("modelling assumption; a real deployment should type errors by each")
print("segment's empirical default rate, not by classification difficulty.")